In [3]:
local_file_path = "./FY2021 대웅출고.xlsx"

In [4]:
import urllib.parse
import os
import pandas as pd
import logging
import io
import re
import csv

from io import BytesIO
from openpyxl import load_workbook
import os
import urllib.parse
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# 로컬 파일 경로에서 bucket_name, full_path, file_name 추출
def get_file_info(local_file_path):
    try:
        # URL 인코딩된 문자열일 경우 디코딩
        full_path = urllib.parse.unquote_plus(local_file_path, encoding='utf-8')
        _, file_name = os.path.split(full_path)
        bucket_name = None  # 로컬이므로 bucket 개념 없음
        logger.info(f"Retrieved file info: full_path={full_path}, file_name={file_name}")
        return bucket_name, full_path, file_name
    except Exception as e:
        logger.error(f"Error getting file info: {e}")
        raise e


# 로컬 파일에서 엑셀 파일 읽기
def get_excel_file(full_path):
    try:
        with open(full_path, "rb") as f:
            excel_data = f.read()
        logger.info(f"Loaded Excel file from local path={full_path}")
        return excel_data
    except Exception as e:
        logger.error(f"Error loading Excel file: {e}")
        raise e


# 엑셀 데이터를 DataFrame으로 변환 후 TSV 형식으로 가공하는 함수
def convert_to_tsv(excel_data):
    try:
        all_sheets = pd.read_excel(
            io.BytesIO(excel_data),
            engine="openpyxl",
            header=1,
            sheet_name=None,
            dtype=str
        )

        df = pd.concat(all_sheets.values(), ignore_index=True)
        logger.info(f"Excel all sheets read into DataFrame with shape {df.shape}")
        tsv_buffer = io.StringIO()
        df.to_csv(
            tsv_buffer,
            sep="\t",
            index=False,
            encoding="utf-8",
            quoting=csv.QUOTE_NONE,
            escapechar="\\"
        )
        logger.info(f"Converted DataFrame to TSV format")
        return tsv_buffer.getvalue()

    except Exception as e:
        logger.error(f"Error converting Excel to TSV: {e}")
        raise

import os
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [5]:
# 변환된 TSV 데이터를 로컬에 저장하는 함수
def save_tsv_to_local(tsv_data, full_path, output_dir="./output"):
    try:
        if tsv_data is None:
            logger.warning("TSV data is None. Skipping save.")
            return
        
        # 디렉토리가 없으면 생성
        os.makedirs(output_dir, exist_ok=True)

        base_name, _ = os.path.splitext(os.path.basename(full_path))
        local_path = os.path.join(output_dir, f"rev_{base_name}.tsv")

        with open(local_path, "w", encoding="utf-8") as f:
            f.write(tsv_data)

        logger.info(f"Saved TSV file locally: {local_path}")
        return local_path
    except Exception as e:
        logger.error(f"Error saving TSV locally: {e}")
        raise e

In [6]:
excel_data = get_excel_file(local_file_path)

In [8]:
# all_sheets = pd.read_excel(
#             excel_data,
#             engine="openpyxl",
#             header=1,
#             sheet_name=None,
#             dtype=str
#         )

In [18]:
import gc
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)
logging.basicConfig(level=logging.INFO)

In [25]:
with pd.ExcelFile(local_file_path, engine="openpyxl") as xls:
    tsv_buffer = io.StringIO()
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet, header=1, dtype=str)
        logger.info(f"Sheet Name / Shape: {sheet} / {df.shape}")

        df.to_csv(
            tsv_buffer,
            sep="\t",
            index=False,
            encoding="utf-16",
            quoting=csv.QUOTE_NONE,
            escapechar="\\",
            lineterminator="\n"
        )

        del df
        gc.collect()

tsv_data = tsv_buffer.getvalue()

INFO:root:Sheet Name / Shape: 2104 / (18023, 24)
INFO:root:Sheet Name / Shape: 2105 / (18325, 24)
INFO:root:Sheet Name / Shape: 2106 / (19345, 24)
INFO:root:Sheet Name / Shape: 2107 / (19284, 24)
INFO:root:Sheet Name / Shape: 2108 / (20745, 24)
INFO:root:Sheet Name / Shape: 2109 / (19602, 24)
INFO:root:Sheet Name / Shape: 2110 / (20069, 24)
INFO:root:Sheet Name / Shape: 2111 / (22213, 25)
INFO:root:Sheet Name / Shape: 2112 / (23287, 24)
INFO:root:Sheet Name / Shape: 2201 / (21517, 24)
INFO:root:Sheet Name / Shape: 2202 / (20402, 24)
INFO:root:Sheet Name / Shape: 2203 / (27463, 24)


In [28]:
local_path = './FY2021 대웅출고.tsv'

with open(local_path, "w", encoding="utf-16") as f:
    f.write(tsv_data)